In [ ]:
! pip install -U ddgs

In [2]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests 
import random

/home/ai-with-rohan/Desktop/LangGraph/myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
def calculator(first_num:float,second_num:float,operation:str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operaitons : add,sub,mul,div
    """
    try:
        if operation=='add':
            result = first_num+second_num
        elif operation=='sub':
            result = first_num-second_num    
        elif operation=='mul':
            result = first_num*second_num    
        elif operation=='div':
            if second_num == 0:
                return {'error':'division by zero is not allowed'}
            result = first_num/second_num 
        else:
            return {'error':f'Unsupported operation {operation}'}
        return {'first_num':first_num,'second_num':second_num,'operation':operation,'result':result}
    except Exception as e:
        return {'error':str(e)}
        

In [3]:
@tool
def get_stock_price(symbol:str)->dict:
    """
    Fetch latest stock price for a given symbol (eg. 'AAPL','TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f""
    response = requests.get(url)
    return response.json()

In [ ]:
tools = [get_stock_price,search_tool,get_stock_price]

llm_with_tools = llm.bind_tools(tools)

In [3]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state:ChatState):
    """
    LLM node that may answer or request a tool call
    """
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}

tool_node = ToolNode(tools)
# executes tool calls

In [ ]:
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)

In [ ]:
graph.add_edge(START,'chat_node')
# If LLM asked for a tool go to ToolNode; else finish
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')

In [ ]:
chatbot = graph.compile()
chatbot

In [ ]:
out = chatbot.invoke({'messages':[HumanMessage(content='hello')]})
print(out['messages'][-1].content)

In [ ]:
out = chatbot.invoke({'messages':[HumanMessage(content='what is 2*3')]})
print(out['messages'][-1].content)

In [ ]:
out = chatbot.invoke({'messages':[HumanMessage(content='What is the stock price of jio digital?')]})
print(out['messages'][-1].content)